# 03 – Baseline dan pembanding S1 sampai S4 (CP3)

- **S1:** Swin-T pralatih, weighted sampling, AdamW 1e-4, wd 0,05, batch 32, maks 30 epoch, henti dini sabar 5.
- **S2:** embedding Swin beku (sekali), SMOTE di dalam lipatan latih, kepala linear.
- **S3:** fitur PyRadiomics → MinMax → LASSO → SMOTE → SVM linear / LDA.
- **S4:** S3 + CTGAN pada ruang fitur (50% data latih). Analisis sensitivitas LASSO 0,01 berlabel eksploratif (D3).

Pelatihan dijalankan sebagai skrip yang diluncurkan dari sel. Tiap job punya `last.pt` (atomik) dan `DONE`; job selesai dilewati. **Jangan ganti seed atau buang lipatan karena hasil jelek.**

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

def _root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'configs' / 'base.yaml').exists():
            return q
    raise FileNotFoundError('configs/base.yaml tidak ditemukan')
ROOT = _root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src.config import load_merged, P
from src.launch import run_module

CFG = 'configs/base.yaml'
cfg = load_merged(CFG)
DEVICES = None      # mis. [0, 1]: satu rantai job per GPU lewat CUDA_VISIBLE_DEVICES (PRD §8)
REPS, FOLDS = None, None   # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.

from src.jobs import launch_scenario, job_status
from src.evaluate import evaluate_scenario, compare_scenarios

MAIN = [('S2', 'configs/s2.yaml', 'src.train_s2'), ('S3_svm', 'configs/s3_svm.yaml', 'src.train_s3'),
        ('S3_lda', 'configs/s3_lda.yaml', 'src.train_s3'), ('S4_svm', 'configs/s4_svm.yaml', 'src.train_s3'),
        ('S4_lda', 'configs/s4_lda.yaml', 'src.train_s3'), ('S1', 'configs/s1.yaml', 'src.train')]   # S1 terakhir (paling lama)
SENS = [('S3_svm_a01', 'configs/s3_svm_a01.yaml', 'src.train_s3'), ('S3_lda_a01', 'configs/s3_lda_a01.yaml', 'src.train_s3'),
        ('S4_svm_a01', 'configs/s4_svm_a01.yaml', 'src.train_s3'), ('S4_lda_a01', 'configs/s4_lda_a01.yaml', 'src.train_s3')]
RUN_SENSITIVITY = True       # analisis sensitivitas D3 (eksploratif)
ALLOW_PARTIAL = REPS is not None or FOLDS is not None   # tabel parsial otomatis ditandai PARSIAL dan tidak memenuhi gerbang

bm = json.loads(P(cfg, 'cp0', 'benchmark.json').read_text())
print('Benchmark CP0:', round(bm['estimated_epoch_minutes'], 1), 'menit/epoch ->', round(bm['estimated_30_epoch_hours_per_job'], 1), 'jam per job S1 (batas atas)')

## 1. Fitur untuk S2 dan S3/S4 (dihitung sekali; dapat dilanjutkan)

In [ ]:
rc = run_module('src.embed', ['--config', 'configs/s2.yaml']); print('embedding, kode keluar:', rc)
rc = run_module('src.radiomics_feats', ['--config', 'configs/s3_svm.yaml']); print('radiomics, kode keluar:', rc)

## 2. Luncurkan job (10 rep × lipatan per skenario)

In [ ]:
for sc, path, module in MAIN:
    print('=== ', sc)
    launch_scenario(path, module, REPS, FOLDS, DEVICES if module == 'src.train' else None)
    display(pd.DataFrame(job_status(load_merged(path), sc)).T)

## 3. Analisis sensitivitas LASSO 0,01 (eksploratif, D3)

In [ ]:
if RUN_SENSITIVITY:
    for sc, path, module in SENS:
        print('=== ', sc)
        launch_scenario(path, module, REPS, FOLDS, None)

## 4. Evaluasi tingkat pasien (AUROC, F1, recall, presisi, akurasi) dengan bootstrap persentil bertingkat 2.000 ulangan

In [ ]:
R = {}
for sc, path, _ in MAIN + (SENS if RUN_SENSITIVITY else []):
    R[sc] = evaluate_scenario(load_merged(path), sc, allow_partial=ALLOW_PARTIAL)
display(pd.concat([r['table'].assign(skenario=sc)[['skenario', 'metric', 'estimate', 'ci_low', 'ci_high', 'job_selesai']] for sc, r in R.items()]))

## 5. Selisih berpasangan S1 − S2 (**eksploratif**; perbandingan utama PRD adalah S6 lawan S1 dan S2)

In [ ]:
display(compare_scenarios(cfg, R['S1'], R['S2'], label='eksploratif'))

## 6. Pemeriksaan checkpoint CP3 dan `manifest.json` (sel terakhir)

In [ ]:
from src.checkpoint import finalize_cp3
report = finalize_cp3(cfg)
assert report['checkpoint'] == 'CP3'